# From market quotes to a swap value

You know what a swap is; this notebook explains how QuantSupport represents one
and connects it to a market. By the end you will have a position PV, a par rate,
a cashflow table, and risk to the original market quotes.

Run the setup in [the examples guide](../../README.md) first, then run all cells in
order. Every notebook in this collection starts from a fresh kernel and loads
its own inputs. No earlier notebook must be executed first.

## The objects you will meet

Think of a valuation as three separate decisions: the contract being valued,
the market observed on a date, and the method used to value the contract.
`Swap` holds contract and position terms. `QuoteStore` holds observations.
`CurveConfiguration` explains which observations build a particular curve.
`PricingContext` constructs those curves and supplies them to the native pricer.
`Request` selects the answers you want; `EvaluationResults` contains them.

We use the repository's illustrative **2026-02-24** market throughout. Loading
files belongs in [the local loading helper](market_data.py); the valuation itself
stays here. `load_market` only deserializes inputs. It does not build curves or
price anything.

In [ ]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date
print(
    "Market date:",
    ref,
)
market.quotes.to_dataframe().head(8)


## Dates and conventions carry financial meaning

`Date` is a calendar date; adding a `Period` gives another date. A date shift
does not itself apply a business-day convention. Use a `Calendar` when that is
the convention you intend. A `DayCounter` turns an accrual interval into a year
fraction. A rate of `0.04` means 4%; `0.0001` is one basis point.

Typed enums make Python contracts readable and discoverable in an editor.
Strings are useful in JSON files, but you do not need to encode your Python
product as a large string-valued dictionary.

In [ ]:
six_months = qs.Period(
    6,
    qs.TimeUnit.Months,
)
calendar = qs.Calendar("UnitedStates(Sofr)")
end = ref + six_months
print(
    "Unadjusted end:",
    end,
)
print(
    "Adjusted end:",
    calendar.adjust(
        end,
        qs.BusinessDayConvention.ModifiedFollowing,
    ),
)
print(
    "Actual/360 accrual:",
    qs.DayCounter.Actual360.year_fraction(
        ref,
        end,
    ),
)
print(
    "Receive / pay signs:",
    qs.Side.LongReceive.sign(),
    qs.Side.PayShort.sign(),
)


## Identify a curve before asking for it

`MarketIndex.SOFR` is a lookup key shared by curve configurations, products,
volatility objects, and rate models. It is not a rate value. The curve below is
built from a deposit and OIS quotes. Those quoted instruments are its *pillars*;
the resulting dated discount factors are its *nodes*.

`SingleCurveCSADiscountPolicy` sets the context's base currency and base index. In this
single-currency example SOFR supplies both projection and discounting. A
multi-curve setup can distinguish those roles.

In [ ]:
print(market.csa_policy)
market.curves[0].to_dict()


## Describe the position

This is a five-year receive-fixed swap with USD 1 million notional. `LongReceive`
receives the fixed leg and pays the floating leg. The Python `Swap` specification
combines the instrument terms and the position information needed to construct
the native trade. An identifier lets you join results back to your trade book.

In [ ]:
swap = qs.Swap(
    identifier="USD-RECEIVE-FIXED-5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=1_000_000,
    fixed_rate=0.04,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    day_counter=qs.DayCounter.Actual360,
    fixed_leg_frequency=qs.Frequency.Semiannual,
    floating_leg_frequency=qs.Frequency.Quarterly,
)
trade = qs.Trade(swap, ref, qs.Side.LongReceive)
trade

## Build once, ask for several answers

First construct `context`, then enter `with context:` to build the
configured market automatically and start the automatic-differentiation
tape used for sensitivities. `evaluate` chooses the native swap pricer.
Ordinary swaps need curves, so we do not attach the volatility inputs here.

Use constructed views such as `context.curve(...)` inside this block. Save
numeric values and pandas tables to inspect later: on exit the context clears
its constructed market and rewinds the tape. Avoid nesting active contexts;
scenario examples below use separate, sequential contexts.

In [ ]:
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
)

with context:
    sofr = context.curve(qs.MarketIndex.SOFR)
    five_year_df = sofr.discount_factor(ref + "5Y")
    nodes = sofr.nodes()
    result = context.evaluate(
        trade,
        [
            qs.Request.Value,
            qs.Request.FairRate,
            qs.Request.Cashflows,
            qs.Request.Sensitivities,
        ],
    )
    cashflows = result.cashflows
    risk = result.sensitivities

pd.Series(
    {
        "position_pv_usd": result.price,
        "par_fixed_rate": result.fair_rate,
        "five_year_discount_factor": five_year_df,
    }
)

## Read the result in its own units

`price` is the signed **position PV**, not a rate or a price per 100. `fair_rate`
is the fixed coupon that makes this swap worth zero on this market. The cashflow
table describes payments; floating amounts are projected using the curve and
are not promises of the rates that will be realized. They are undiscounted
amounts, so summing them does not give the PV.

Results only contain the outputs their pricer supplies. For example, this swap
has no bond yield-to-maturity. Request support is product-specific; see
[valuation catalog](../README.md).

In [ ]:
cashflows[["payment_date", "leg_index", "type", "amount", "currency"]].head(10)


The sensitivity table maps the PV back to the original quote identifiers.
`value` is a derivative per **one full unit of the input quote**, not already a
basis-point risk. All inputs below are decimal interest rates, so multiplying
by `1e-4` estimates the USD P&L for a +1 bp move in each quote separately.
The sign is retained: this is a signed quote DV01, not an absolute duration.

In [ ]:
quote_risk = risk.assign(pnl_for_plus_1bp_usd=risk["value"] * 1e-4)
quote_risk.sort_values("pnl_for_plus_1bp_usd")


You now have the common workflow used throughout the collection. Continue with
[valuation catalog](../README.md) to see how the requests change,
[curves and volatility](../../volatilitysurface/curves_and_volatility.ipynb) to change the market, or
[portfolio risk](../../sensitivity/portfolio_risk.ipynb) to aggregate positions and check AAD
against full revaluation. [Simulations](../../hullwhite/simulations.ipynb) and
[XVA](../../cva/xva.ipynb) explain what changes when future market states are needed.